<a href="https://colab.research.google.com/github/nmwiley808/csci198-Music-Intelligence-with-Deep-Learning-Senior-Project/blob/main/notebooks/02%20-%20Audio%20Preprocessing%20%26%20Feature%20Extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 – Audio Preprocessing & Feature Extraction

This notebook converts raw audio from:

- GTZAN
- FMA Small
- MagnaTagATune

into standardized log-mel spectrogram features.

All audio is:
- Resampled to 22,050 Hz
- Standardized to 30 seconds
- Converted to log-mel spectrograms

Processed arrays are saved to `data/processed/` for model training.

In [28]:
# Mount Drive & Setup
from google.colab import drive
drive.mount('/content/drive')

import os, time
import numpy as np
import pandas as pd
import librosa
import tqdm

PROJECT_PATH = "/content/drive/MyDrive/csci198/csci198-Music-Intelligence-with-Deep-Learning-Senior-Project"
os.chdir(PROJECT_PATH)
print("Working directory:", os.getcwd())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working directory: /content/drive/MyDrive/csci198/csci198-Music-Intelligence-with-Deep-Learning-Senior-Project


In [8]:
# Parameters
SAMPLE_RATE       = 22050
DURATION          = 30
SAMPLES_PER_TRACK = SAMPLE_RATE * DURATION
N_MELS            = 128
FMA_SAVE_EVERY    = 500  # save a batch every N tracks to protect against crashes

os.makedirs("data/processed", exist_ok=True)

In [9]:
# Shared Helpers
def load_audio(path):
    try:
        y, _ = librosa.load(path, sr=SAMPLE_RATE)
        if len(y) < SAMPLES_PER_TRACK:
            y = np.pad(y, (0, SAMPLES_PER_TRACK - len(y)))
        else:
            y = y[:SAMPLES_PER_TRACK]
        return y
    except Exception:
        return None

def extract_log_mel(y):
    mel = librosa.feature.melspectrogram(y=y, sr=SAMPLE_RATE, n_mels=N_MELS)
    return librosa.power_to_db(mel, ref=np.max)

In [10]:
# Process GTZAN
X_GTZAN = "data/processed/X_gtzan.npy"
Y_GTZAN = "data/processed/y_gtzan.npy"

if os.path.exists(X_GTZAN):
    print("GTZAN cache found – skipping.")
    X_gtzan = np.load(X_GTZAN)
    y_gtzan = np.load(Y_GTZAN, allow_pickle=True)
    print("Shape:", X_gtzan.shape)
else:
    gtzan_root = "data/raw/gtzan/Data/genres_original"
    X_gtzan, y_gtzan = [], []
    t0 = time.time()

    for label in sorted(os.listdir(gtzan_root)):
        genre_dir = os.path.join(gtzan_root, label)
        if not os.path.isdir(genre_dir):
            continue
        for fname in tqdm.tqdm(sorted(os.listdir(genre_dir)), desc=label):
            audio = load_audio(os.path.join(genre_dir, fname))
            if audio is None:
                continue
            X_gtzan.append(extract_log_mel(audio))
            y_gtzan.append(label)

    X_gtzan = np.array(X_gtzan)
    y_gtzan = np.array(y_gtzan)
    np.save(X_GTZAN, X_gtzan)
    np.save(Y_GTZAN, y_gtzan)
    print(f"GTZAN saved: {X_gtzan.shape}  ({round(time.time()-t0,1)} s)")

jazz:  54%|█████▍    | 54/100 [00:06<00:02, 16.47it/s]/tmp/ipykernel_21864/3226120861.py:4: UserWarning: PySoundFile failed. Trying audioread instead.
  y, _ = librosa.load(path, sr=SAMPLE_RATE)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
rock: 100%|██████████| 100/100 [00:07<00:00, 13.58it/s]


GTZAN saved: (999, 128, 1292)  (124.3 s)


In [11]:
# Build FMA Genre Map
# FMA files are named by track ID (000002.mp3 = track 2).
# Genre labels live in tracks.csv, NOT in the folder names.
tracks_csv = "data/raw/fma/fma_metadata/tracks.csv"

if not os.path.exists(tracks_csv):
    raise FileNotFoundError(
        f"{tracks_csv} not found.\n"
        "Download fma_metadata.zip from https://github.com/mdeff/fma "
        "and extract to data/raw/fma/fma_metadata/"
    )

tracks = pd.read_csv(tracks_csv, index_col=0, header=[0, 1])
small  = tracks[tracks['set', 'subset'] == 'small']
fma_genre_map = small['track', 'genre_top'].dropna().to_dict()

print(f"Genre map built: {len(fma_genre_map)} tracks")
print("Genres:", sorted(set(fma_genre_map.values())))

Genre map built: 8000 tracks
Genres: ['Electronic', 'Experimental', 'Folk', 'Hip-Hop', 'Instrumental', 'International', 'Pop', 'Rock']


In [12]:
# Process FMA Small
fma_root  = "data/raw/fma/fma_small"
BATCH_DIR = "data/processed/fma_batches"
X_FMA     = "data/processed/X_fma.npy"
Y_FMA     = "data/processed/y_fma.npy"
os.makedirs(BATCH_DIR, exist_ok=True)

if os.path.exists(X_FMA):
    print("FMA cache found – skipping.")
    X_fma = np.load(X_FMA)
    y_fma = np.load(Y_FMA, allow_pickle=True)
    print("Shape:", X_fma.shape)
else:
    # Collect every mp3 in sorted order so resumption is deterministic
    all_mp3s = sorted([
        os.path.join(r, f)
        for r, _, files in os.walk(fma_root)
        for f in files if f.endswith(".mp3")
    ])
    print(f"{len(all_mp3s)} mp3 files found")

    # How many tracks are already safely saved in previous batches?
    saved_batches = sorted([f for f in os.listdir(BATCH_DIR) if f.endswith(".npz")])
    already_done  = sum(
        len(np.load(os.path.join(BATCH_DIR, b), allow_pickle=True)['X'])
        for b in saved_batches
    )
    print(f"Resuming from file index {already_done} ({len(saved_batches)} batches already saved)")

    X_buf, y_buf = [], []
    batch_idx    = len(saved_batches)
    t0           = time.time()

    for path in tqdm.tqdm(all_mp3s[already_done:], desc="FMA Small"):
        try:
            track_id = int(os.path.splitext(os.path.basename(path))[0])
        except ValueError:
            continue
        if track_id not in fma_genre_map:
            continue

        audio = load_audio(path)
        if audio is None:
            continue

        X_buf.append(extract_log_mel(audio))
        y_buf.append(fma_genre_map[track_id])

        # Save every FMA_SAVE_EVERY tracks
        if len(X_buf) >= FMA_SAVE_EVERY:
            out = os.path.join(BATCH_DIR, f"batch_{batch_idx:04d}.npz")
            np.savez(out, X=np.array(X_buf), y=np.array(y_buf))
            print(f"  Batch {batch_idx} saved ({len(X_buf)} tracks)")
            batch_idx += 1
            X_buf, y_buf = [], []

    # Flush remaining tracks
    if X_buf:
        out = os.path.join(BATCH_DIR, f"batch_{batch_idx:04d}.npz")
        np.savez(out, X=np.array(X_buf), y=np.array(y_buf))
        print(f"  Final batch {batch_idx} saved ({len(X_buf)} tracks)")

    # Merge all batches into final arrays
    print("Merging batches...")
    all_batches = sorted([f for f in os.listdir(BATCH_DIR) if f.endswith(".npz")])
    Xs = [np.load(os.path.join(BATCH_DIR, b), allow_pickle=True)['X'] for b in all_batches]
    ys = [np.load(os.path.join(BATCH_DIR, b), allow_pickle=True)['y'] for b in all_batches]
    X_fma = np.concatenate(Xs)
    y_fma = np.concatenate(ys)

    np.save(X_FMA, X_fma)
    np.save(Y_FMA, y_fma)
    print(f"FMA saved: {X_fma.shape}  ({round(time.time()-t0,1)} s)")
    print("Labels:", sorted(set(y_fma)))

8001 mp3 files found
Resuming from file index 0 (0 batches already saved)


FMA Small:   6%|▌         | 500/8001 [05:02<1:41:18,  1.23it/s]

  Batch 0 saved (500 tracks)


FMA Small:  12%|█▏        | 1000/8001 [10:00<2:18:30,  1.19s/it]

  Batch 1 saved (500 tracks)


...

  Batch 13 saved (500 tracks)


FMA Small:  94%|█████████▍| 7504/8001 [1:16:01<07:59,  1.04it/s]

  Batch 14 saved (500 tracks)


FMA Small: 100%|██████████| 8001/8001 [1:21:13<00:00,  1.64it/s]


  Final batch 15 saved (497 tracks)
Merging batches...
FMA saved: (7997, 128, 1292)  (4908.4 s)
Labels: [np.str_('Electronic'), np.str_('Experimental'), np.str_('Folk'), np.str_('Hip-Hop'), np.str_('Instrumental'), np.str_('International'), np.str_('Pop'), np.str_('Rock')]


In [31]:
# Process MagnaTagATune
X_MAGNA = "data/processed/X_magna.npy"
Y_MAGNA = "data/processed/y_magna.npy"

if os.path.exists(X_MAGNA):
    print("MagnaTagATune cache found – skipping.")
    X_magna = np.load(X_MAGNA)
    y_magna = np.load(Y_MAGNA, allow_pickle=True)
    print("Shape:", X_magna.shape)
else:
    magna_root = "data/raw/magnatagatune/Data"
    anno_path  = os.path.join(magna_root, "annotations_final.csv")

    anno     = pd.read_csv(anno_path, sep='\t')
    tag_cols = [c for c in anno.columns if c not in ['clip_id', 'mp3_path']]
    anno     = anno.set_index('mp3_path')

    X_magna, y_magna = [], []
    t0 = time.time()

    for root, _, files in os.walk(magna_root):
        for fname in tqdm.tqdm(sorted(files), desc=os.path.relpath(root, magna_root)):
            if not fname.endswith(".mp3"):
                continue
            rel = os.path.relpath(os.path.join(root, fname), magna_root)
            if rel not in anno.index:
                continue
            audio = load_audio(os.path.join(root, fname))
            if audio is None:
                continue
            X_magna.append(extract_log_mel(audio))
            y_magna.append(anno.loc[rel, tag_cols].values.astype(int))

    X_magna = np.array(X_magna)
    y_magna = np.array(y_magna)
    np.save(X_MAGNA, X_magna)
    np.save(Y_MAGNA, y_magna)
    print(f"MagnaTagATune saved: {X_magna.shape}  ({round(time.time()-t0,1)} s)")

8:  35%|███▍      | 395/1144 [00:50<01:01, 12.21it/s]/tmp/ipykernel_21864/3226120861.py:4: UserWarning: PySoundFile failed. Trying audioread instead.
  y, _ = librosa.load(path, sr=SAMPLE_RATE)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
6:  68%|██████▊   | 1211/1788 [02:14<00:48, 11.79it/s]/tmp/ipykernel_21864/3226120861.py:4: UserWarning: PySoundFile failed. Trying audioread instead.
  y, _ = librosa.load(path, sr=SAMPLE_RATE)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
9:   7%|▋         | 132/1773 [00:35<02:20, 11.71it/s]/tmp/ipykernel_21864

MagnaTagATune saved: (25860, 128, 1292)  (3317.6 s)


In [32]:
# Summary
print("=== Preprocessing Complete ===")
print(f"GTZAN:          {X_gtzan.shape}")
print(f"FMA Small:      {X_fma.shape}  | genres: {sorted(set(y_fma))}")
print(f"MagnaTagATune:  {X_magna.shape}  | {y_magna.shape[1]} binary tag columns")

=== Preprocessing Complete ===
GTZAN:          (999, 128, 1292)
FMA Small:      (7997, 128, 1292)  | genres: [np.str_('Electronic'), np.str_('Experimental'), np.str_('Folk'), np.str_('Hip-Hop'), np.str_('Instrumental'), np.str_('International'), np.str_('Pop'), np.str_('Rock')]
MagnaTagATune:  (25860, 128, 1292)  | 188 binary tag columns
